In [6]:
import pandas as pd
import numpy as np

In [7]:
lookback = 50
assets = ['VIX', 'VTI', 'DBC', 'AGG']
csv_files = {
    'VIX': 'VIX_historical_OHLCV.csv',
    'VTI': 'VTI_historical_OHLCV.csv',
    'DBC': 'DBC_historical_OHLCV.csv',
    'AGG': 'AGG_historical_OHLCV.csv'
}

In [8]:
dfs = []
for asset in assets:
    df = pd.read_csv(csv_files[asset], parse_dates=['Date'])
    df = df[['Date', 'Close']].rename(columns={'Close': f'Close_{asset}'})
    dfs.append(df)

In [9]:
data = dfs[0]
for df in dfs[1:]:
    data = pd.merge(data, df, on='Date', how='inner')


In [10]:
for asset in assets:
    data[f'Return_{asset}'] = data[f'Close_{asset}'].pct_change()

In [11]:
data = data.dropna().reset_index(drop=True)


* Text from paper-

"the input features of one asset
can be its past prices and returns with a dimension of (k; 2) where k represents the lookback window.
By stacking features across all assets, the dimension of the resulting input would be (k; 2 * n)."

In [18]:
X = []
Y = []
dates = []

for i in range(lookback, len(data)):
    # Prepare features (X)
    features = []
    for asset in assets:
        prices = data[f'Close_{asset}'].iloc[i-lookback:i].values.reshape(-1, 1)
        returns = data[f'Return_{asset}'].iloc[i-lookback:i].values.reshape(-1, 1)
        asset_features = np.hstack([prices, returns])  
        features.append(asset_features)
    
    sample = np.hstack(features)
    X.append(sample)
    
    # Prepare target returns (Y)
    target_returns = []
    for asset in assets:
        # Get the next period's return
        next_return = data[f'Return_{asset}'].iloc[i]
        target_returns.append(next_return)
    
    Y.append(target_returns)
    dates.append(data['Date'].iloc[i])

X = np.array(X)
Y = np.array(Y)

print("X shape:", X.shape)  # (num_samples, 50, 8)
print("Y shape:", Y.shape)  # (num_samples, 4)

# Save both X and Y
np.save('X_data.npy', X)
np.save('Y_data.npy', Y)

X shape: (3492, 50, 8)
Y shape: (3492, 4)
